<a href="https://colab.research.google.com/github/fidlarsyn/grokking-Deep-Learning-Chapter-7-to-10/blob/main/10_Neural_Learning_About_Edges_and_Corners.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Reusing Weights in Multiple Places**

Masalah terbesar dalam jaringan saraf tiruan adalah *overfitting*, yaitu ketika jaringan menghafal *noise* pada dataset alih-alih mempelajari abstraksi yang berguna untuk generalisasi. Solusi efektif untuk mengatasi *overfitting* tanpa mengorbankan daya ekspresi model adalah menerapkan **struktur**—yaitu memilih secara selektif untuk menggunakan ulang bobot (*reusing weights*) di banyak posisi berbeda ketika kita yakin pola yang sama perlu dideteksi di berbagai lokasi gambar. Teknik penggunaan ulang bobot yang paling terkenal dan luas digunakan ini disebut **konvolusi** (*convolution*), dan ketika digunakan sebagai layer disebut **convolutional layer**.

# **The Convolutional Layer**

Dibandingkan menggunakan layer linier besar yang padat (*dense*) di mana setiap input terhubung ke setiap output, *convolutional layer* menggunakan banyak layer linier berukuran sangat kecil (biasanya memiliki kurang dari 25 input dan 1 output) yang dipasang di setiap posisi gambar. Mini-layer ini disebut **convolutional kernel**.
* **Mekanisme Pergeseran (** **Sliding** **)**: Kernel 3×3 melakukan prediksi di lokasi saat ini, bergeser 1 piksel ke kanan, memprediksi lagi, dan seterusnya hingga memindai seluruh gambar.
* **Operasi Pooling**: Ketika beberapa kernel menghasilkan matriks prediksi, hasilnya digabungkan melalui operasi *pooling*. Metode yang paling populer adalah **max pooling**, di mana pada setiap posisi dipilih nilai maksimum dari output seluruh kernel untuk diteruskan ke layer berikutnya.

In [ ]:
import numpy as np, sys
np.random.seed(1)

from keras.datasets import mnist

# 1. Load & Preprocess Data MNIST
(x_train, y_train), (x_test, y_test) = mnist.load_data()

images, labels = (x_train[0:1000].reshape(1000, 28 * 28) / 255, y_train[0:1000])
one_hot_labels = np.zeros((len(labels), 10))
for i, l in enumerate(labels):
    one_hot_labels[i][l] = 1
labels = one_hot_labels

test_images = x_test.reshape(len(x_test), 28 * 28) / 255
test_labels = np.zeros((len(y_test), 10))
for i, l in enumerate(y_test):
    test_labels[i][l] = 1

# 2. Aktivasi & Derivatif
def tanh(x):
    return np.tanh(x)

def tanh2deriv(output):
    return 1 - (output ** 2)

def softmax(x):
    temp = np.exp(x)
    return temp / np.sum(temp, axis=1, keepdims=True)

# 3. Parameter Jaringan & Inisialisasi Kernel
alpha, iterations = (2, 300)
pixels_per_image, num_labels = (784, 10)
batch_size = 128

input_rows, input_cols = (28, 28)
kernel_rows, kernel_cols = (3, 3)
num_kernels = 16

hidden_size = ((input_rows - kernel_rows) * (input_cols - kernel_cols)) * num_kernels

kernels = 0.02 * np.random.random((kernel_rows * kernel_cols, num_kernels)) - 0.01
weights_1_2 = 0.2 * np.random.random((hidden_size, num_labels)) - 0.1

def get_image_section(layer, row_from, row_to, col_from, col_to):
    section = layer[:, row_from:row_to, col_from:col_to]
    return section.reshape(-1, 1, row_to - row_from, col_to - col_from)

# 4. Training Loop (CNN Forward & Backprop)
for j in range(iterations):
    correct_cnt = 0
    for i in range(int(len(images) / batch_size)):
        batch_start, batch_end = ((i * batch_size), ((i + 1) * batch_size))
        layer_0 = images[batch_start:batch_end]
        layer_0 = layer_0.reshape(layer_0.shape, 28, 28)

        # Ekstraksi seluruh sub-bagian wilayah 3x3
        sects = list()
        for row_start in range(layer_0.shape[12] - kernel_rows):
            for col_start in range(layer_0.shape[13] - kernel_cols):
                sect = get_image_section(layer_0,
                                         row_start,
                                         row_start + kernel_rows,
                                         col_start,
                                         col_start + kernel_cols)
                sects.append(sect)
        expanded_input = np.concatenate(sects, axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es * es[12], -1)

        # Forward Propagation Konvolusi
        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es, -1))
        layer_2 = np.dot(layer_1, weights_1_2)

        for k in range(batch_size):
            label_offset = batch_start + k
            layer_2_code = np.argmax(layer_2[k:k+1])
            labels_code = np.argmax(labels[label_offset:label_offset+1])
            correct_cnt += int(layer_2_code == labels_code)

        # Backpropagation
        layer_2_delta = (labels[batch_start:batch_end] - layer_2) / (batch_size * layer_2.shape)
        layer_1_delta = layer_2_delta.dot(weights_1_2.T) * tanh2deriv(layer_1)

        # Pembaruan Bobot Layer 1-2 & Kernel Konvolusi
        weights_1_2 += alpha * layer_1.T.dot(layer_2_delta)

        l1d_reshape = layer_1_delta.reshape(es, es[12], -1)
        k_update = flattened_input.T.dot(l1d_reshape.reshape(es * es[12], -1))
        kernels += alpha * k_update

    test_correct_cnt = 0
    for i in range(len(test_images)):
        layer_0 = test_images[i:i+1]
        layer_0 = layer_0.reshape(1, 28, 28)
        sects = list()
        for row_start in range(layer_0.shape[12] - kernel_rows):
            for col_start in range(layer_0.shape[13] - kernel_cols):
                sect = get_image_section(layer_0,
                                         row_start,
                                         row_start + kernel_rows,
                                         col_start,
                                         col_start + kernel_cols)
                sects.append(sect)
        expanded_input = np.concatenate(sects, axis=1)
        es = expanded_input.shape
        flattened_input = expanded_input.reshape(es * es[12], -1)
        kernel_output = flattened_input.dot(kernels)
        layer_1 = tanh(kernel_output.reshape(es, -1))
        layer_2 = np.dot(layer_1, weights_1_2)

        test_correct_cnt += int(np.argmax(layer_2) == np.argmax(test_labels[i:i+1]))

    if (j % 10 == 0 or j == iterations - 1):
        sys.stdout.write("\\n" + \\
            "I:" + str(j) + \\
            " Test-Acc:" + str(test_correct_cnt / float(len(test_images))) + \\
            " Train-Acc:" + str(correct_cnt / float(len(images))))

# **A Simple Implementation in NumPy**

* Implementasi konvolusi dalam NumPy dilakukan dengan mengekstraksi sub-bagian 2D (*subregions*) dari *batch* gambar menggunakan fungsi bantuan `get_image_section()`. Setiap sub-bagian diperlakukan seolah-olah merupakan gambar kecil tersendiri. Seluruh sub-bagian tersebut diratakan (*flattened*) dan dikalikan dengan matriks kernel via perkalian titik (`flattened_input.dot(kernels)`) untuk memproses prediksi seluruh posisi secara efisien dan cepat.

In [ ]:
def get_image_section(layer, row_from, row_to, col_from, col_to):
    section = layer[:, row_from:row_to, col_from:col_to]
    return section.reshape(-1, 1, row_to - row_from, col_to - col_from)